# Differential Power Analysis

## Inspecting the Measurement Data

Let us first inspect the raw measurement data (or *traces*). The trace data is not included in the notebook code, it's actually quite big. In order to obtain the data, execute the cell below by entering it and pressing shift+enter. You can also open a terminal in the root directory of this exercise and type `make traces`

In [ ]:
%%bash
make traces

It may take a while to download and extract the trace data. Once finished, try and execute the following code. It plots the raw trace data of the first measurement in the database.

In [ ]:
%matplotlib notebook

import matplotlib.pyplot as plt
from dpa.traces import Trace_dir
 
# Open directory containing the measurement data
trace_dir = Trace_dir('secmatv1_2006_04_0809_2000')

# Get the first measurement and show a nice plot
measurement = trace_dir.__next__()
plt.plot(measurement.trace)

You should see a trace ranging over about 20,000 samples. If this is not the case, check if you dowloaded the trace data or go and ask your teacher for help. Can you figure out where in the trace the DES encryption takes place? Don't worry, this is not important to make the attack work.

The class `Trace_dir` is used to iterate over all the measurement files in the trace directory. It returns an object of type `Measurement` (see file `dpa/traces.py` for details). A measurement object `m` is composed of the raw trace data `m.trace` and the plain text message `m.plaintext` that has been encrypted to obtain the trace.

In [ ]:
from dpa.traces import Trace_dir
 
# Open directory containing the measurement data
trace_dir = Trace_dir('secmatv1_2006_04_0809_2000')

# Get the first measurement and show its plain text message
measurement = trace_dir.__next__()
print( measurement.plaintext )

The plaintext is a list of eight bytes, corresponding to the block size of 64 bits for DES.

## Observation Window

The original traces have a length of 20,000 samples. Since the analysis is quite data intensive, it is a good idea to reduce the *observation window* to a smaller part of the traces, where we know that the leakage takes place. For now, run the code in the cell below. If you want to further reduce the observation window later, change the values and rerun the cell:

In [ ]:
# Set the observation window
start_sample = 4000
end_sample = 7999
num_samples = end_sample - start_sample + 1

If you want, you can look at a plot of the observation window for the first trace in the database:

In [ ]:
%matplotlib notebook

import matplotlib.pyplot as plt
from dpa.traces import Trace_dir
 
# Open directory containing the measurement data
trace_dir = Trace_dir('secmatv1_2006_04_0809_2000')

# Get the first measurement and plot the observation window
measurement = trace_dir.__next__()
plt.plot(measurement.trace[start_sample:end_sample])

## The Leakage Model

You are lucky, we already provide the leakage model for you. It is included in the file `dpa/des.py`(html/des.html). Open the file in your favorite editor or [follow this link](html/des.html#leakage_function) and scroll down to the definition of the function `des_first_round_hd_4bits()`. You do not need to understand all the details of the function, for now just look at the arguments it takes:
  * `sbox` : the index of the sbox under attack
  * `plaintext` : the plaintext being encrypted (this is a known input from the measurements data base)
  * `key_hypothesis` : a hypothesis on the 6 bits of the first round subkey corresponding to the sbox inputs
  
As the name of the function suggests, it returns the *hamming distance* corresponding to the state register update of the four bits affected by the selected sbox. We expect therefore outputs between 0 and 4. 

Let's test the leakage function. In the example below, we will take some dummy plaintext message (a list of 8 bytes) and compute all the predicted hamming distance values for the 64 possible key hypotheses for sbox 3. Afterwards, we will plot the distribution of the resulting leakage values.

In [ ]:
%matplotlib notebook

import matplotlib.pylab as plt
from dpa.des import *

plaintext = [1, 2, 3, 4, 5, 6, 7, 8]
sbox = 3

# store the number of occurences of hamming distance values
hd_count = [0 for i in range(5)]

# calculate leakage values for all 64 hypotheses
for key_hypothesis in range(64):
    l = des_first_round_hd_4bits(sbox, plaintext, key_hypothesis)
    hd_count[l] += 1
    
# plot the output distribution
plt.bar(range(5), hd_count)

You should see a symmetric bar plot with the center of the distribution around the value 2. This is not surprising, if you think about the hamming distance of two four-bit variables, there are 16 different values per variable, thus 256 different combinations. Out of these, there are only 16 possibilities to get the value 0 (both variables are identical) and 4 (the variables are complementary), respectively. There are a lot more combinations leading to 1, 2, or 3, the value 2 being the most frequent.

## Implementing the Attack

So let's go and implement the attack. Let us recall first the DPA algorithm as seen in the lecture: 

<img src="img/algo.png" width="500" style="float:left"/>

Here, $\mathcal{M}$ is the leakage function and $W_i$ are the traces. So basically for each hypothesis $S_H$, we start by *partitioning the traces* (lines 3 to 6). To do so, we compute the value of the leakage function (line 4) for each trace and use the computed value to classify the trace into one of the sets $E_{\mathcal{M}=j}$ ($j$ being the value we just computed). Since the value range of our leakage function is $\{0, \dots, 4\}$, we will have 5 partitions. In the next step, we compute a *mean trace* for each partition (lines 7 to 9). These mean traces are then combined to a *differential trace* (line 10). We will end up with one differential trace per hypothesis (in our case 64 differential traces). The only information we will keep for each differential trace -- and which is used as *distinguisher* -- is the maximum sample (line 11). The good hypothesis is then the one that gives the highest maximum sample (line 13).  

### 1. Partitioning the Traces

So this is what we would do if we would take the pseudo code above literally:
 1. Iterate over all hypotheses
 2. Iterate over all traces
 3. Comute the leakage function
 4. Put the current trace in one of 5 sets
 5. Once done, go on to compute the mean traces
 
Unfortunately, in terms of efficient code, there are two problems (or opportunities to do better) here:
 * The trace data base is quite big, and file operations are slow. Iterating 64 times over all the traces is not ideal.
 * Keeping sets of individual traces around seems unneccessary, since each set will only be used to compute a mean trace afterwards.

The solution for the first item is exchanging the order of the loops: We will rather iterate over all traces, and for each trace, iterate over all the hyotheses. In this way, we only need to read each trace once from the disk, and we can discard it afterwards. This should keep the memory footprint of our implementation small.

For the second point, remember how we compute the mean trace for a partition $E_{\mathcal{M}=j}$ (we will use $E_j$ as a shortcut in the following):

$\overline{W}_j(t) = \frac{1}{|E_j|} \cdot \sum_{W \in E_j} W(t)$

In words, each sample in the mean trace is computed as the sum of the corresponding samples in the set, divided by the number of traces in the set. We can actually avoid keeping sets of individual traces around by *computing the sum in place*, i.e. by keeping only an *accumulator* that provides the sum of all the traces. In order to compute the mean trace, all that is left to do is the division. Thus, we also need to remember the *number of traces* in each "set", since this information is otherwise lost.

OK, let's define the necessary data structures. For the partitions (think accumulators), we basically need a big table: One accumulated trace for each sbox for each key hypothesis for each possible leakage value. For the trace itself, we will use numpy arrays, which are much faster that Python lists for computations. Here comes our table:

In [ ]:
import numpy

# One accumulated trace for each sbox, key hypothesis, and hamming distance
accumulator = [[[numpy.zeros(num_samples) for hd in range(5)] for key_hyp in range(64)] for sbox in range(8)]

# We also need to keep track of the number of traces accumulated in each "set"
acc_num_traces = [[[0] * 5 for key_hyp in range(64)] for sbox in range(8)]

Note that in the list comprehension, the indices are listed in reverse order: The sbox is the "outermost" dimension, while the hamming distance comes last. So if you access the table, the order is
```python
accumulator[sbox][key_hyp][hd]
```

It's finally time for you to take over: Complete the function defintion below. As described above, we will iterate over all traces (or rather `num_traces` given as an argument), over all sboxes, over all key hypotheses, calculate the leakage function (given the key hypothesis and the plaintext associated with the trace) and accumulate the trace on the accumulator index by the calculated leakage value:

In [ ]:
# TODO: Complete the function definition

from tqdm import tqdm_notebook as tqdm
from tqdm import tqdm, trange
from dpa.traces import Trace_dir
import dpa.des as des


def accumulate(num_traces):
    global accumulator, acc_num_traces
    
    # Read trace database, keeping num_traces 
    trace_dir = Trace_dir('secmatv1_2006_04_0809_2000', num_traces)
    
    # For each trace
    for measurement in tqdm(trace_dir, total=num_traces):
        
        # For each sbox
        for sbox in range(8):
            
            # For each key hypothesis
            for key_hyp in range(64):
                
                # TODO: put your code here
                pass

You can test your function with the code below, which will just call `accumulate`. If there are no syntax errors, and you are satisfied with your code, you can go on.

In [ ]:
# Test code for accumulate
accumulate(100)

Some implementation hints: 
  * The traces (`measurement.trace`) are numpy arrays
  * You can apply arithmetic operations (such as addition) on numpy arrays without iterating over all the elements
  * You can do *slicing* on numpy arrays just like on ordinary lists: `array[n:m]` returns a sub-array with the elements between the indices $n$ and $m-1$
  * you can multiply a scalar value with a numpy array, resulting in the array where each entry is multiplied with the scalar

### 2. Computing the Mean Traces

Now that we have classified and accumulated all the traces, we can compute the mean traces. There will be one mean trace for each sbox for each key hypothesis and for each leakage value (hamming distance). Here is our data structure:

In [ ]:
# mean_traces[sbox][key_hyp][hd]
mean_traces = [[[numpy.zeros(num_samples) for hd in range(5)] for key_hyp in range(64)] for sbox in range(8)]

Now it's up to you to complete the function definition below. The function `compute_mean_traces` should iterate over all sboxes, key_hypotheses, and leakage values, and compute the mean traces from the corresponding accumulator and number of traces per class:

In [ ]:
def compute_mean_traces():
    global mean_traces, accumulator, acc_num_traces
    
    # TODO: Complete the function definition
    pass

### 3. Computing the Differential Traces

At this point, we have five mean traces $\overline{W_0} \dots \overline{W_4}$ for each sbox and for each key hypothesis. The next step is to combine them to one differential trace per key hypothesis. Here is the equation from the lecture that defines the weighted sum used to compute a differential trace:

$W_\Delta(t) = -2\cdot\overline{W_0}(t) -\overline{W_1}(t) + \overline{W_3}(t) + 2\cdot\overline{W_4}(t)$

If you wonder about the weights, scroll up to the plot of the distribution of the hamming distance. The distribution is centered around the value 2. We give negative weight to the mean traces for which we expect a lower power consumption ($\overline{W_0}$ and $\overline{W_1}$) and positive weight to the ones for which we expect a higher power consumption ($\overline{W_3}$ and $\overline{W_4}$). The one in the center of the distribution does not contribute to the weighted sum, its weight is zero. For any bad hypothesis (and for any sample which does not correspond to the time of the leakage), the classification of the traces should be more or less random. Applying the above equation to such a partitioning will result in values close to zero. For the good hypothesis and the time of the leakage, we expect to see a spike in the differential trace.

We will end up with one differential trace for each sbox and each key hypothesis. Here is the data structure used to store the differential traces:

In [ ]:
# diff_traces[sbox][key_hyp]
diff_traces = [[numpy.zeros(num_samples) for key_hyp in range(64)] for sbox in range(8)]

Complete the function that computes the differential traces:

In [ ]:
def compute_diff_traces():
    global diff_traces
    
    # TODO: complete the function definition
    pass

### 4. Putting it all Together

We are almost done. Now that we have one differential trace per sbox per key hypothesis, we need to find for each sbox the one differential trace with the highest peak value. So basically, this corresponds to the lines 11 and 13 of the pseudo code:

$\mathcal{D}(S_H) \leftarrow max_t\;W_\Delta$(t)

$S_{good} \leftarrow arg\,max\;\mathcal{D}$

The first operation means that we will compress each differential trace to only one value, keeping the maximum sample only. With this, we will end up with one value per sbox and per key_hypothesis. The second operation finds for each sbox the good key hypothesis (the one leading to the maximum peak value).

Here is our data structures holding the peak values and the good key hypotheses:

In [ ]:
# diff_peak_value[sbox][key_hyp]
diff_peak_values = [[0.0] * 64 for sbox in range(8)]

# guessed_key[sbox]
guessed_key = [0 for sbox in range(8)]

Complete the following two functions, realizing the operations discussed above:

In [ ]:
def compute_diff_peak_values():
    global diff_peak_values
    
    # TODO: complete this function
    pass

In [ ]:
def compute_guessed_key():
    global guessed_key
    
    # TODO: complete this function
    pass

### 5. Running the Complete Algorithm

If you have completed the sections 1 to 4 above, then it's time to run the complete Differential Power Analysis. Let's give it a try:

In [ ]:
def dpa(num_traces):
    print ("Partitioning traces...")
    accumulate(num_traces)
    print ("Computing mean traces...")
    compute_mean_traces()
    print ("Computing differential traces...")
    compute_diff_traces()
    print ("Retreiving guessed key...")
    compute_diff_peak_values()
    compute_guessed_key()
    
    print("\nThe guessed key is:")
    hex_key = ['{:02x}'.format(x) for x in guessed_key]
    print (' '.join(hex_key))   

In [ ]:
dpa(10) # Need at least 500 for a correct result

The correct key is `38 0B 3B 26 00 0D 19 37`. Did you find it?

### Visualizing the result

After running the DPA, you can execute the following code in order to see the result for one of the SBoxes. It shows the differential traces for all 64 hypotheses, with the trace corresponding to the best hypothesis highlighted in red.

In [ ]:
%matplotlib notebook
import matplotlib.pylab as plt

sbox = 5

for key_hyp in range(64):
    if key_hyp == guessed_key[sbox]: 
        plt.plot(diff_traces[sbox][key_hyp], color='red', linewidth=2)
    else:
        plt.plot(diff_traces[sbox][key_hyp], color='blue', linewidth=0.5)